# Power BI Input Preparation

Creates reproducible Power BI source files from the Task 1 and Task 2B notebook outputs.

## 1. Load notebook outputs

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = root / "output"
output_dir.mkdir(exist_ok=True)

def find_file(name):
    return output_dir / name

task1_file = find_file("task1_analysis_ready_trips.csv")
spatial_file = find_file("task2b_spatial_enriched_trips.csv")

if task1_file is None or spatial_file is None:
    raise FileNotFoundError("Run Task 1 and Task 2B first.")

task1 = pd.read_csv(task1_file)
spatial = pd.read_csv(spatial_file)

print(f"Task 1 trips: {len(task1):,}")
print(f"Spatially enriched trips: {len(spatial):,}")

Task 1 trips: 6,631
Spatially enriched trips: 6,173


## 2. Build full trip-level Power BI table

In [2]:
# Stable join fields
for frame in (task1, spatial):
    frame["caseid"] = frame["caseid"].astype("string")
    frame["trip_number_numeric"] = pd.to_numeric(frame["trip_number_numeric"], errors="coerce")

spatial_fields = [
    "caseid", "trip_number_numeric",
    "origin_lta", "destination_lta", "od_scope"
]
spatial_lookup = spatial[spatial_fields].drop_duplicates(
    ["caseid", "trip_number_numeric"]
)

fact = task1.merge(
    spatial_lookup,
    on=["caseid", "trip_number_numeric"],
    how="left",
    validate="one_to_one"
)

print(f"Power BI fact rows: {len(fact):,}")
print(f"Rows with spatial assignment: {fact['origin_lta'].notna().sum():,}")

Power BI fact rows: 6,631
Rows with spatial assignment: 6,173


**Check:** The left join preserves all Task 1 trips; spatial fields remain missing where OD coordinates were unavailable.

## 3. Correct duplicate-review flag

In [3]:
dup_fields = [
    "caseid", "Destination eastings", "Destination northings",
    "Start time - first hour", "Travel Time (minutes)", "Main mode"
]

complete_dup = fact[dup_fields].notna().all(axis=1)
fact["potential_duplicate_flag"] = (
    complete_dup & fact.duplicated(dup_fields, keep=False)
)

print("Potential duplicate records:", int(fact["potential_duplicate_flag"].sum()))

Potential duplicate records: 12


## 4. Add analytical fields used in Power BI

In [4]:
MODE_GROUP = {
    "Car/ van (as the driver)": "Car / van",
    "Car/ van (as a passenger)": "Car / van",
    "Walking": "Active / wheeling",
    "Pedal cycle": "Active / wheeling",
    "Electric cycle (e-bike)": "Active / wheeling",
    "Mobility scooter": "Active / wheeling",
    "Hire e-bike/ e-scooter": "Micromobility",
    "Rail": "Public transport",
    "Public bus service": "Public transport",
    "Other coach (e.g. long distance coaches)": "Public transport",
    "Ferry": "Public transport",
    "Private bus/ coach (e.g. school service, other private service)": "Private bus / coach",
    "Taxi/ minicab": "Taxi / private hire",
    "Motorcycle/ moped": "Motorcycle / moped",
    "Something else": "Other / unspecified",
}

DISTANCE_LABELS = {
    1: "Under 1 mile",
    2: "1 to <2 miles",
    3: "2 to <5 miles",
    4: "5 to <10 miles",
    5: "10 to <25 miles",
    6: "25 to <50 miles",
    7: "50 to <100 miles",
    8: "100+ miles",
}

fact["mode_group"] = fact["Main mode"].map(MODE_GROUP)

distance_cat = pd.to_numeric(fact["Distance category"], errors="coerce")
fact["distance_band"] = distance_cat.map(DISTANCE_LABELS)
fact["distance_band_order"] = distance_cat

fact["start_hour"] = pd.to_numeric(
    fact["Start time - first hour"], errors="coerce"
)

fact["time_band"] = pd.cut(
    fact["start_hour"],
    bins=[-0.1, 6, 10, 15, 19, 24],
    labels=[
        "00:00-06:00", "06:00-10:00", "10:00-15:00",
        "15:00-19:00", "19:00-24:00"
    ],
    right=False
).astype("string")

fact["day_type"] = np.select(
    [
        fact["Day of week"].isin(["Saturday", "Sunday"]),
        fact["Day of week"].notna()
    ],
    ["Weekend", "Weekday"],
    default=None
)

car_modes = [
    "Car/ van (as the driver)",
    "Car/ van (as a passenger)"
]

distance_valid = (
    fact["distance_analysis_valid"]
    .astype("string").str.lower().eq("true")
)

fact["is_car_van"] = fact["Main mode"].isin(car_modes)
fact["is_car_driver"] = fact["Main mode"].eq("Car/ van (as the driver)")
fact["is_car_passenger"] = fact["Main mode"].eq("Car/ van (as a passenger)")
fact["is_active_wheeling"] = fact["mode_group"].eq("Active / wheeling")
fact["is_public_transport"] = fact["mode_group"].eq("Public transport")

fact["is_short_trip"] = distance_valid & distance_cat.isin([1, 2, 3])
fact["is_mid_distance_trip"] = distance_valid & distance_cat.isin([4, 5])
fact["is_short_car"] = fact["is_car_van"] & fact["is_short_trip"]
fact["is_mid_car"] = fact["is_car_van"] & fact["is_mid_distance_trip"]
fact["is_mid_public_transport"] = (
    fact["is_public_transport"] & fact["is_mid_distance_trip"]
)
# Observed trip count for each respondent in the supplied trip extract
fact["respondent_trip_count"] = (
    fact.groupby("caseid")["caseid"]
    .transform("size")
)

print("Unmapped source modes:",
      fact.loc[fact["Main mode"].notna() & fact["mode_group"].isna(),
               "Main mode"].nunique())

Unmapped source modes: 0


In [5]:
respondent_trip_validation = (
    fact[["caseid", "respondent_trip_count"]]
    .drop_duplicates()
    ["respondent_trip_count"]
    .value_counts()
    .sort_index()
)

print("Observed trip count distribution:")
print(respondent_trip_validation)

print(
    "\nTotal respondents:",
    respondent_trip_validation.sum()
)

Observed trip count distribution:
respondent_trip_count
1    2625
2     859
3     348
4     143
5      60
6      62
Name: count, dtype: Int64

Total respondents: 4097


## 5. Add validated trip-chain membership

In [6]:
seq = fact.copy()
seq["_row_id"] = seq.index
seq["trip_num"] = pd.to_numeric(
    seq["trip_number_numeric"], errors="coerce"
)
seq = seq.sort_values(["caseid", "trip_num"])

for c in [
    "trip_num",
    "Origin eastings", "Origin northings",
    "Destination eastings", "Destination northings",
    "_row_id"
]:
    seq[f"next_{c}"] = seq.groupby("caseid")[c].shift(-1)

seq["consecutive"] = seq["next_trip_num"].eq(seq["trip_num"] + 1)
seq["od_gap_m"] = np.hypot(
    pd.to_numeric(seq["Destination eastings"], errors="coerce")
    - pd.to_numeric(seq["next_Origin eastings"], errors="coerce"),
    pd.to_numeric(seq["Destination northings"], errors="coerce")
    - pd.to_numeric(seq["next_Origin northings"], errors="coerce")
)

pairs = seq[
    seq["consecutive"] & seq["od_gap_m"].le(100)
].copy()

chain_rows = (
    set(pairs["_row_id"].astype(int))
    | set(pairs["next__row_id"].dropna().astype(int))
)

fact["validated_chain_member"] = fact.index.isin(chain_rows)

fact["short_car_chain_status"] = np.select(
    [
        fact["is_short_car"] & fact["validated_chain_member"],
        fact["is_short_car"]
    ],
    [
        "Part of validated chain",
        "Not identified in validated chain"
    ],
    default=None
)

print("Validated chain-member trips:",
      int(fact["validated_chain_member"].sum()))

Validated chain-member trips: 1758


**Assumption:** Validated chaining uses consecutive trip numbers and destination-to-next-origin continuity within 100 m.

## 6. Create Power BI dimension tables

In [7]:
dim_mode = (
    fact[["Main mode", "mode_group"]]
    .dropna(subset=["Main mode"])
    .drop_duplicates()
    .sort_values("Main mode")
    .reset_index(drop=True)
)
dim_mode.insert(0, "ModeKey", range(1, len(dim_mode) + 1))

dim_purpose = (
    fact[["Purpose_category"]]
    .dropna()
    .drop_duplicates()
    .sort_values("Purpose_category")
    .reset_index(drop=True)
)
dim_purpose.insert(0, "PurposeKey", range(1, len(dim_purpose) + 1))

dim_distance = pd.DataFrame({
    "DistanceKey": range(1, 9),
    "DistanceBand": [DISTANCE_LABELS[i] for i in range(1, 9)],
    "SortOrder": range(1, 9)
})

date_values = pd.to_datetime(fact["Date"], errors="coerce").dropna().drop_duplicates()
dim_date = pd.DataFrame({"Date": sorted(date_values)})
dim_date["DateKey"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["DayOfWeek"] = dim_date["Date"].dt.day_name()
dim_date["DayOrder"] = dim_date["DayOfWeek"].map({
    "Monday": 1, "Tuesday": 2, "Wednesday": 3,
    "Thursday": 4, "Friday": 5, "Saturday": 6, "Sunday": 7
})
dim_date["DayType"] = np.where(
    dim_date["DayOfWeek"].isin(["Saturday", "Sunday"]),
    "Weekend", "Weekday"
)

dim_time = pd.DataFrame({"StartHour": range(24)})
dim_time["TimeKey"] = dim_time["StartHour"] + 1
dim_time["HourLabel"] = dim_time["StartHour"].map(lambda x: f"{x:02d}:00")
dim_time["TimeBand"] = pd.cut(
    dim_time["StartHour"],
    bins=[-0.1, 6, 10, 15, 19, 24],
    labels=[
        "00:00-06:00", "06:00-10:00", "10:00-15:00",
        "15:00-19:00", "19:00-24:00"
    ],
    right=False
).astype("string")
dim_time["TimeBandOrder"] = dim_time["TimeBand"].map({
    "00:00-06:00": 1,
    "06:00-10:00": 2,
    "10:00-15:00": 3,
    "15:00-19:00": 4,
    "19:00-24:00": 5
})

origin_values = sorted(fact["origin_lta"].dropna().unique())
dim_origin_lta = pd.DataFrame({
    "OriginLTAKey": range(1, len(origin_values) + 1),
    "OriginLTA": origin_values
})

destination_values = sorted(fact["destination_lta"].dropna().unique())
dim_destination_lta = pd.DataFrame({
    "DestinationLTAKey": range(1, len(destination_values) + 1),
    "DestinationLTA": destination_values
})

print(
    len(dim_mode), "modes |",
    len(dim_purpose), "purposes |",
    len(dim_origin_lta), "origin geographies"
)

15 modes | 7 purposes | 17 origin geographies


## 7. Add dimension keys to fact table

In [8]:
fact = fact.merge(
    dim_mode,
    on=["Main mode", "mode_group"],
    how="left"
)

fact = fact.merge(
    dim_purpose,
    on="Purpose_category",
    how="left"
)

fact["DistanceKey"] = distance_cat.where(distance_cat.between(1, 8))

fact["DateKey"] = (
    pd.to_datetime(fact["Date"], errors="coerce")
    .dt.strftime("%Y%m%d")
)
fact["DateKey"] = pd.to_numeric(fact["DateKey"], errors="coerce")

fact["TimeKey"] = fact["start_hour"].where(
    fact["start_hour"].between(0, 23)
) + 1

fact = fact.merge(
    dim_origin_lta,
    left_on="origin_lta",
    right_on="OriginLTA",
    how="left"
).drop(columns=["OriginLTA"])

fact = fact.merge(
    dim_destination_lta,
    left_on="destination_lta",
    right_on="DestinationLTA",
    how="left"
).drop(columns=["DestinationLTA"])

print("Fact rows after keys:", len(fact))

Fact rows after keys: 6631


## 8. Export Power BI source files

In [9]:
powerbi_dir = output_dir / "powerbi"
powerbi_dir.mkdir(exist_ok=True)

files = {
    "fact_trips.csv": fact,
    "dim_mode.csv": dim_mode,
    "dim_purpose.csv": dim_purpose,
    "dim_distance.csv": dim_distance,
    "dim_date.csv": dim_date,
    "dim_time.csv": dim_time,
    "dim_origin_lta.csv": dim_origin_lta,
    "dim_destination_lta.csv": dim_destination_lta,
}

for name, data in files.items():
    data.to_csv(powerbi_dir / name, index=False)

print("Created:")
for name in files:
    print(" -", powerbi_dir / name)

Created:
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\fact_trips.csv
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\dim_mode.csv
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\dim_purpose.csv
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\dim_distance.csv
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\dim_date.csv
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\dim_time.csv
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\dim_origin_lta.csv
 - c:\Users\nehav\OneDrive\Desktop\NEHA\transport-regional-travel-analysis\output\powerbi\dim_destination_lta.csv


## 9. Validation

In [10]:
validation = pd.DataFrame({
    "Check": [
        "Fact rows",
        "Respondents",
        "Potential duplicate records",
        "Rows with origin LTA",
        "Rows with destination LTA",
        "Unmapped modes",
        "Validated chain-member trips"
    ],
    "Value": [
        len(fact),
        fact["caseid"].nunique(),
        int(fact["potential_duplicate_flag"].sum()),
        int(fact["origin_lta"].notna().sum()),
        int(fact["destination_lta"].notna().sum()),
        int(fact.loc[
            fact["Main mode"].notna() & fact["mode_group"].isna(),
            "Main mode"
        ].nunique()),
        int(fact["validated_chain_member"].sum())
    ]
})

validation

,Check,Value
0,Fact rows,6631
1,Respondents,4097
2,Potential duplicate records,12
3,Rows with origin LTA,6173
4,Rows with destination LTA,6173
5,Unmapped modes,0
6,Validated chain-member trips,1758


In [11]:
validation.to_csv(
    powerbi_dir / "validation_summary.csv",
    index=False
)

print("Validation summary saved.")

Validation summary saved.
